Classificatore di Sequence con LSTM

In [ ]:
import torch
import torch.nn as nn

class LSTMClassifier(nn.Module):
    """
    Classificatore di sequenze basato su LSTM a strati multipli
    """
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super(LSTMClassifier, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Modulo LSTM principale
        self.lstm = nn.LSTM(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True, # input: (batch, T, d_x)
            dropout = dropout if num_layers > 1 else 0.0,
            bidirectional = False
        )

        # Strato di classificazione finale
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes),
        )

    # ATTENZIONE: Ora "forward" è allineato correttamente con "__init__"!
    def forward(self, x, lengths=None):
        """
        Parametri
        -------
        x : tensore di forma (batch, T, d_x)
        lengths: tensore 1-D con lunghezze reali delle sequenze nel batch (opzionale per padding)

        Restituisce logits: tensore di forma (batch, num_classes)
        """
        batch_size = x.size(0)

        # Inizializzazione degli strati a zero
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=x.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=x.device)

        # Gestione di sequenze con padding
        if lengths is not None:
            x = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)

        # Passaggio in avanti attraverso la LSTM
        out, (hn, cn) = self.lstm(x, (h0, c0))

        if lengths is not None:
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)

        # Utilizzo dell'ultimo stato nascosto (strato finale)
        # hn ha la forma (num_layers, batch, hidden_size)
        last_hidden = hn[-1]    # (batch, hidden_size)

        logits = self.classifier(last_hidden)
        return logits

Modello GRU per Serie Temporali

In [ ]:
import torch
import torch . nn as nn

class GRURegressor ( nn .Module ):
    """
    Regressore per serie temporali basato su GRU bidirezionale .
    Parametri
    −−−−−−−−−
    input_size : numero divariabili ( canali ) della serie temporale
    hidden_size : dimensione dello stato nascosto
    num_layers : strati GRU impilati
    output_size : dimensione dell’ output ( es.! per regressione scalare )
    dropout : probabilita’ di dropout
    """

    def __init__( self , input_size , hidden_size ,
                   num_layers = 2 , output_size =1 , dropout = 0.2 ):
        super ( GRURegressor , self ).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = True
        self.num_layers = num_layers    # Bidirezionale

        self.gru = nn.GRU(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True,
            dropout = dropout if num_layers > 1 else 0.0,
            bidirectional = self.bidirectional
        )

        # hidden state finale concatena le due direzioni
        self.classifier = nn.Linear(
            hidden_size * self.num_directions,
            output_size,
        )

Modello GRU per Serie Temporali

In [ ]:
import torch
import torch.nn as nn

class GRURegressor(nn.Module):
    def __init__( self , input_size , hidden_size , num_layers=2, output_size=1 , dropout=0.2 ):
        super (GRURegressor, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = True
        self.num_directions = 2 # Bidirezionale

        self.gru = nn.GRU(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True,
            dropout = dropout if num_layers > 1 else 0.0,
            bidirectional = self.bidirectional
        )

        # Hidden state finale concatena le due direzioni
        self.regressor = nn.Linear(
            hidden_size * self.num_directions,
            output_size,
        )

    def forward(self, x):
        # Out : (batch, T, hidden_size * num_directions)
        # hn : (num_layers * num_directions, batch, hidden_size)
        out, hn = self.gru(x)

        # Concatenazione degli stati finali delle due direzioni
        # hn[-2]: Ultimo strato, direzione forward
        # hn[-1]: Ultimo strato, direzione backward
        last_hidden = torch.cat([hn[-2], hn[-1]], dim = -1)

        pred = self.regressor(last_hidden)
        return pred


Addestramenot e Gradient Clipping

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Iperparametri
MAX_GRAD_NORM = 5.0     # Soglia per il gradient clipping
NUM_EPOCHS = 30
LR = 1e-3

# Inizializzazione del modello, perdita e ottimizzatore
model = LSTMClassifier(input_size=64, hidden_size=128, num_layers=2, num_classes=10)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# Messo una volta sola
model.train()

for epoch in range(NUM_EPOCHS):
    for x_batch, y_batch, lengths in train_loader:
        optimizer.zero_grad()

        logits = model(x_batch, lengths)
        loss = criterion(logits, y_batch)

        loss.backward()

        # Gradient clipping: Aggiunto il trattino basso "_" alla fine di norm_
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)

        optimizer.step()



NameError: name 'LSTMClassifier' is not defined